In [22]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for script execution
import matplotlib.pyplot as plt
import seaborn as sns

from dbm301.config import RAW_DATA_DIR, FIGURES_DIR, PROCESSED_DATA_DIR ,INTERIM_DATA_DIR 

# Tạo thư mục output nếu chưa có
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Thu vien da san sang.')
print(f'Figures dir: {FIGURES_DIR}')

Thu vien da san sang.
Figures dir: C:\Users\Admin\Documents\DBM301\Assignment1_de4\dbm301_asm_group3_de4\reports\figures


In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Đọc file gốc từ thư mục data/raw/
df = pd.read_csv(RAW_DATA_DIR / 'bank-full.csv', sep=';')

print(f'So quan sat : {df.shape[0]:,}')
print(f'So thuoc tinh: {df.shape[1]}')
df.head()
print(f"Tổng số dòng ban đầu: {len(df):,} dòng")

So quan sat : 45,211
So thuoc tinh: 17
Tổng số dòng ban đầu: 45,211 dòng


In [24]:
#thuc hien phan tang co seed
df_exploration, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['y'])
print(f"Tập 80% (Exploration): {len(df_exploration):,} dòng")
print(f"Tập 20% (Test)       : {len(df_test):,} dòng")

Tập 80% (Exploration): 36,168 dòng
Tập 20% (Test)       : 9,043 dòng


In [25]:
# Xem tỷ lệ phần trăm y=yes và y=no ở cả 2 tập
print("Tỷ lệ ở tập 80%:")
print(df_exploration['y'].value_counts(normalize=True) * 100)

print("\nTỷ lệ ở tập 20%:")
print(df_test['y'].value_counts(normalize=True) * 100)

Tỷ lệ ở tập 80%:
y
no    88.3018
yes   11.6982
Name: proportion, dtype: float64

Tỷ lệ ở tập 20%:
y
no    88.3003
yes   11.6997
Name: proportion, dtype: float64


In [26]:
# Lưu ra file CSV (dùng dấu chấm phẩy ; để đồng bộ với file gốc)
df_exploration.to_csv(INTERIM_DATA_DIR / 'exploration_80.csv', sep=';', index=False)
df_test.to_csv(INTERIM_DATA_DIR / 'test_20.csv', sep=';', index=False)

print("Đã lưu thành công 2 file vào data/interim/!")

Đã lưu thành công 2 file vào data/interim/!


### NHẬN XÉT GIỚI HẠN CỦA DỮ LIỆU KHI CHIA TẬP (TASK 2.5):

1. **Giới hạn 1 — Về thứ tự thời gian:** 
   - Dữ liệu thu thập từ tháng 5/2008 đến tháng 11/2010. 
   - Việc chia ngẫu nhiên làm xáo trộn thời gian: các cuộc gọi của năm 2010 bị đưa vào tập học (80%), còn cuộc gọi năm 2008 lại rơi vào tập kiểm tra (20%). 
   - Điều này vi phạm nguyên tắc dòng thời gian thực tế (lấy dữ liệu tương lai đi kiểm tra cho quá khứ).

2. **Giới hạn 2 — Về việc thiếu mã ID khách hàng:** 
   - Dữ liệu không có cột định danh khách hàng (`customer_id`). Mỗi dòng chỉ là một cuộc gọi đơn lẻ. 
   - Một khách hàng có thể được gọi nhiều lần và vô tình bị chia lẫn: cuộc gọi thứ nhất rơi vào tập 80%, cuộc gọi thứ hai rơi vào tập 20%. 
   - Điều này làm cho tập 20% bị rò rỉ thông tin cá nhân của người đã xuất hiện ở tập 80%, làm giảm tính khách quan khi kiểm tra luật.